In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Policy gradient II: what weights each step?

## Where this session fits

> **The limitation we start from: the raw return is a very noisy weight, so the estimate is very noisy (high variance)**

- **The question:** what can weight each step instead of the raw return, without changing the expected gradient, or at a price we can state?
- **Position: Part B (policy gradient), session 2 of its 4 sessions**
    - B1 gave the theorem: $\nabla_\theta J$ is an average of terms $\nabla_\theta \log \pi_\theta$, each multiplied by a **weight** (a single number)
    - the raw return was the weight
    - "weighting the returns" and "weighting the gradient terms" are the **same thing**: the return *is* the number that multiplies the gradient term
    - this session fills slot 1 with a baseline and the advantage, and meets **REINFORCE**, the theorem estimated by sampling
- **The three slots** of the plan from B1 (theorem): the parts of the gradient formula that the methods change
    1. **slot 1, the weight** $w$ on each gradient term **← this session, and the next**
    2. **slot 2, how far** one update moves the policy
    3. **slot 3, where the baseline comes from** (inside slot 1) **← this session, in part**
- **Series goal:** choose good actions by trial and error, from small tables to robots and language models

**By the end you can**

1. say where the noise in the weight comes from
2. define a baseline and the advantage, and say why a baseline that does not depend on the action does not bias the gradient
3. say why the advantage compares one action with the state's average return
4. write the advantage form of the gradient, and read REINFORCE as the theorem estimated by sampling, with a baseline
5. place REINFORCE in the slot table, and say why a critic is the next step


**Where this session sits in Part B**

<img src="images/partB_arc_b2.png" width=95%>


# Variance returns: where the noise in the weight comes from

**Why this section?** Bias and variance were defined for the *targets* of Monte Carlo and TD. The same two words now describe the *weight* in the policy gradient.

> **The weight is unbiased but has high variance, for three reasons. Each reason has a fix, and the fixes are this session.**

- recall (from sampling)
    - **bias:** how far the average of an estimate is from the truth
    - **variance:** how much the estimate fluctuates from sample to sample
- the weight in the theorem, a sampled return, is **unbiased** (B1 theorem) but has **high variance**, for three reasons
    - **the whole return is in it**
        - the raw return says how much was collected, not how good the action was
        - an action is good or bad *relative to the returns of the other actions* in that state
        - if all returns have the same sign, no single return can be called good or bad
        - only the *differences* between returns carry signal, and the shared part adds noise
    - **it is one sample:** a single sampled episode, whose rewards depend on the environment's randomness and on the policy's own sampling
    - **no per-step credit:** the same number weights every step, so with sparse rewards we cannot say *which* step deserves the credit

**The three fixes**

1. subtract something every action shares: a **baseline** (answers the first reason, the shared part)
2. give each step its own weight: **credit assignment**, the **advantage**
    - answers the third reason, no per-step credit
    - uses only the rewards that follow a step, or a critic (item 3)
3. replace part of the sampled return by an estimate from a **critic**
    - a critic is a learned network that estimates the expected return from a state, as in Part A
    - answers the second reason, one noisy sample
    - as a baseline it lowers variance with no bias
    - as a bootstrap it lowers variance much further, at the price of bias, as TD did
    - it also helps with credit
    - this session covers fixes 1 and 2; the critic is the next session


## A baseline: subtract what every action would get anyway

**Why this section?** The first reason: the weight contains a part shared by all actions. Remove it and the signal that remains is what the action added.

> **A baseline $b(\state)$ depends on the state but not on the action, and is subtracted from the weight. The expected gradient is unchanged; a good baseline lowers the spread.**

- suppose every return is positive
    - the theorem pushes *up* every sampled action, in proportion to the raw return
    - but an action is good only if its return beats the returns of the other actions
    - most of each return would have come anyway, whichever action was taken
- a **baseline** $b(\state)$ is a number that depends on the state but **not on the action**, subtracted from the weight
    - for example, the average return from that state
- the **advantage** is the return relative to the baseline
    - $\advseq = \rlret - b(\state)$: positive if its return beat the other actions' typical return, negative if it fell short

**The same estimator with and without a baseline**

**What is plotted: the one-sample gradient estimate for logit $0$ (weight $\times$ score), not the returns.**

- left panel: the weight is the raw return
- right panel: the weight is the return minus the baseline $b$
- the score is $+2/3$ when action $0$ is drawn and $-1/3$ otherwise

> **The right bump on the left is the draws of action $0$, the lowest-return arm. The raw return pushes it *up* because every return is positive; the baseline removes that push.**

- the dark draws are action $0$ (one third of the samples)
    - raw return: about $+1.3$, an *up* push on a below-average arm
    - with the baseline: about $-0.3$, a *down* push
- the baseline subtracts $b$ times the score from every sample, so the mean is unchanged
- the spread falls by a factor of about $2.4$ ($1.16$ to $0.49$)

<img src="images/baseline_variance.png" width=75%>


**Why a baseline does not bias the gradient**

> **A baseline leaves the expected gradient unchanged, provided it does not depend on the action.**

- the score function $\nabla_\theta \log \pi_\theta$ has zero mean (shown in the appendix to B1 (theorem))

$$
\E_{\actseq \sim \pi_\theta}\bigl[ \nabla_\theta \log \pi_\theta(\actseq \mid \state) \, b(\state) \bigr] = b(\state) \, \E\bigl[ \nabla_\theta \log \pi_\theta(\actseq \mid \state) \bigr] = 0
$$

- this needs $b$ **not to depend on the action**: then it comes out of the expectation
- so subtracting $b$ leaves the expected gradient unchanged
- what changes is the variance
- a good baseline cancels the part of the return that all actions share
- what remains is the return *relative to the other actions*, which is what separates good from bad

**The exact advantage: one action against the average**

> **The advantage subtracts the state's *average* return from the return of one *specific* action: $\advseq = \actvalfun_\pi(\state, \act) - \statevalfun_\pi(\state)$.**

- $\actvalfun_\pi(\state, \act)$ is the expected return when we take this **one action** $\act$ in $\state$, then follow $\pi$
- $\statevalfun_\pi(\state)$ is the expected return **of the state**, averaged over the actions the policy may take
    - $\statevalfun_\pi(\state) = \sum_\act \pi_\theta(\act \mid \state) \, \actvalfun_\pi(\state, \act)$
- so $\advseq$ is positive when this action is better than the policy's average action, and negative when worse
- the sampled $\rlret_\tt - b$ is a sample of $\actvalfun_\pi - b$, because $\E[\rlret_\tt \mid \state, \act] = \actvalfun_\pi(\state, \act)$
    - it is a sample of the exact advantage when $b = \statevalfun_\pi(\state)$
- caution: a baseline computed from the same sample (for example a group mean that includes it) depends slightly on the action
- the leave-one-out mean avoids this


## Credit assignment: one weight per step

**Why this section?** The third reason (no per-step credit): one weight for a whole episode cannot say which step deserves the credit.

> **One weight per episode cannot say which step deserves the credit. The fix is an advantage per step.**

- the episode-return variant gives *every* step the same weight
    - a language model writes a 100-word masterpiece and mistypes the last character
        - the final reward is low, and *all* steps are punished
        - though one step is to blame
    - with sparse rewards we cannot tell good steps from bad ones, so learning is slow (high variance)
- the fix: an **advantage per step**, $\advseq_{\traj,\tt}$, which sets the force and direction of each step's gradient
    - two mechanisms give per-step credit
        - use only the rewards that *follow* a step: the return-to-go $\rlret_{\tt,\traj}$ instead of $\rlret_{0,\traj}$ (it helps only when rewards arrive at several steps)
        - a **critic**, a learned estimate of the value of each step's state (next session)
    - a baseline $b(\state)$ alone reduces noise but does not assign credit


**What the advantage does to the gradient**

<table>
<thead><tr><th nowrap><div align="left">Advantage</div></th><th nowrap><div align="left">Effect on the gradient</div></th><th nowrap><div align="left">Result</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left"><b>positive</b> ($\advseq_{\traj,\tt} > 0$)</div></td><td nowrap><div align="left">positive multiplier</div></td><td nowrap><div align="left"><b>accelerate</b>: better than usual, make it more likely</div></td></tr>
<tr><td nowrap><div align="left"><b>negative</b> ($\advseq_{\traj,\tt} < 0$)</div></td><td nowrap><div align="left">flips the sign</div></td><td nowrap><div align="left"><b>brake</b>: below expectation, make it less likely</div></td></tr>
<tr><td nowrap><div align="left"><b>about zero</b></div></td><td nowrap><div align="left">nullifies the gradient</div></td><td nowrap><div align="left"><b>maintain</b>: as expected, nothing to learn</div></td></tr>
</tbody></table>

- advantage-based methods are the basis of modern policy gradient
- one well-known early example is A3C (Mnih et al., 2016)


## The unified form: slot 1 is the advantage

**Why this section?** With a weight per step, the methods differ only in how they compute it. One formula makes them comparable.

> **Every method of Part B estimates this gradient from sampled episodes. They differ in the weight $\advseq$ (slot 1), in how far each update moves (slot 2), and in where the baseline comes from (slot 3).**

$$
\nabla_\theta J(\theta) = \E_{\traj \sim \pi_\theta} \left[ \sum_{\tt=0}^{|\traj|-1} \nabla_\theta \log \pi_\theta(\actseq_{\traj,\tt} \mid \stateseq_{\traj,\tt}) \; \advseq_{\traj,\tt} \right]
$$

- the weight of B1 (theorem)'s slot 1 is now $\advseq_{\traj,\tt}$
- what each method chooses
    - **REINFORCE** (this session): the theorem estimated by sampling, with a baseline from past episodes
    - **actor-critic** (B3 critic): the advantage from a *learned* value function, a critic
    - **PPO** (B4 step size): the same gradient, with a clipped update (slot 2) and a critic
    - **GRPO** (Part C): PPO's update with a baseline from a group of episodes
- we drop the factor $\disc^\tt$ of the exact discounted gradient, as most implementations do
    - it is $1$ when $\disc = 1$
    - this is the common practical form, not exactly the discounted gradient


# REINFORCE: the theorem, estimated by sampling

**Why this section?** The simplest method with an advantage, and the most direct use of the theorem. It is the reference point for what follows.

> **REINFORCE is the policy gradient theorem estimated from sampled episodes: weight each $\nabla_\theta \log \pi_\theta$ by the return-to-go minus a baseline. Nothing else is added.**

- it is the sampled estimator of B1 (theorem), with one change: a baseline
- the weight is $\advseq_{\traj,\tt} = \rlret_{\tt,\traj} - b$
    - $\rlret_{\tt,\traj}$ is the return-to-go: the rewards that follow step $\tt$
    - $b$ is a baseline, often a moving average of the returns of past episodes
- it needs **no critic**: nothing is learned except the policy
- "Monte Carlo" is the same sense as in sampling: the full return of a complete sampled episode

**The REINFORCE loop**

- repeat
    1. sample an episode $\traj$ by following $\pi_\theta$
    2. compute the return-to-go $\rlret_{\tt,\traj}$ of every step $\tt$
    3. set the weight $\advseq_{\traj,\tt} = \rlret_{\tt,\traj} - b$
    4. step: $\theta \leftarrow \theta + \alpha \sum_\tt \advseq_{\traj,\tt} \, \nabla_\theta \log \pi_\theta(\actseq_{\traj,\tt} \mid \stateseq_{\traj,\tt})$
    5. update the baseline $b$, after the step has used it
- the baseline used in steps 3 and 4 is its value *before* this episode updates it
    - then $b$ cannot depend on this episode's actions
    - this is the condition of the baseline slide: no bias

**The moving-average baseline, and a worked step**

- a common baseline: a **moving average** of the returns of the episodes so far
    - $b_{\text{new}} = (1 - w)\, b_{\text{old}} + w \, \rlret_{0,\traj}$, with a small weight $w$ (not the learning rate $\alpha$)
- a worked step, with $w = 0.1$ and $b = 5$
    - an episode returns $8$: the weight is $8 - 5 = +3$, so its actions are pushed *up*
    - then $b \leftarrow 0.9 \cdot 5 + 0.1 \cdot 8 = 5.3$
    - the next episode returns $4$: the weight is $4 - 5.3 = -1.3$, so its actions are pushed *down*
- when the only reward comes at the end and $\disc = 1$, every step has the same return-to-go
    - so every step gets the same weight: no per-step credit
    - per-step credit with an end-only reward needs a critic (next session)


**The same loop for a language model: one training prompt at a time**

- for each training prompt $\x$
    - generate an output $\y$ from the policy $\pi_\theta$
    - its reward $\rew$ comes from an evaluator
    - the baseline is the current moving average $b$ (a constant for this episode)
    - gradient step on $\theta$ in the direction $(\rew - b)\, \nabla_\theta \log \pi_\theta(\y \mid \x)$
    - then update $b$ by the rule above


**REINFORCE: what it gives and what it costs**

> **REINFORCE is unbiased and needs only sampled episodes, but its weight is one noisy sample.**

<table>
<thead><tr><th nowrap><div align="left"></div></th><th nowrap><div align="left">What it gives</div></th><th nowrap><div align="left">What it costs</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left"><b>bias and variance</b></div></td><td nowrap><div align="left">unbiased, if the baseline ignores the action</div></td><td nowrap><div align="left">high variance: one sampled return, the whole tail of random rewards</div></td></tr>
<tr><td nowrap><div align="left"><b>machinery</b></div></td><td nowrap><div align="left">nothing is learned except the policy: no critic</div></td><td nowrap><div align="left">the baseline is a single number that ignores the state</div></td></tr>
<tr><td nowrap><div align="left"><b>credit</b></div></td><td nowrap><div align="left">per-step credit through the return-to-go, when rewards come at every step</div></td><td nowrap><div align="left">with one end reward, every step gets the same weight</div></td></tr>
<tr><td nowrap><div align="left"><b>data</b></div></td><td nowrap><div align="left">any reward, even a score from a program; no model of the environment</div></td><td nowrap><div align="left">complete episodes, and on-policy: after one update the batch came from the old policy, so it is thrown away</div></td></tr>
</tbody></table>

**Each cost points to a later session**

<table>
<thead><tr><th nowrap><div align="left">The cost</div></th><th nowrap><div align="left">The fix</div></th><th nowrap><div align="left">Session</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">a baseline that ignores the state</div></td><td nowrap><div align="left">a state-dependent baseline, an estimate of $\statevalfun_\pi(\state)$: the first role of a critic</div></td><td nowrap><div align="left">3</div></td></tr>
<tr><td nowrap><div align="left">the noisy tail of one sampled return</div></td><td nowrap><div align="left">replace the tail by a critic's estimate: the second role of a critic</div></td><td nowrap><div align="left">3</div></td></tr>
<tr><td nowrap><div align="left">each batch is used once: after the update its episodes came from the old policy</div></td><td nowrap><div align="left">reuse the batch for several updates: a probability ratio corrects for the old policy, and clipping keeps the correction trustworthy</div></td><td nowrap><div align="left">4</div></td></tr>
<tr><td nowrap><div align="left">a critic is a second network to train</div></td><td nowrap><div align="left">a baseline from a group of episodes of the same prompt</div></td><td nowrap><div align="left">Part C</div></td></tr>
</tbody></table>


## The slot table: review and preview

**Why this section?** B1 (theorem) previewed three slots. Here is what we have filled so far (review), and what the next sessions change (preview).

<table>
<thead><tr><th nowrap><div align="left">Method</div></th><th nowrap><div align="left">Slot 1: weight $\advseq$</div></th><th nowrap><div align="left">Slot 2: how far</div></th><th nowrap><div align="left">Slot 3: baseline source (inside slot 1)</div></th></tr></thead>
<tbody>
<tr><td colspan="4"><div align="left"><b>Review: filled so far</b></div></td></tr>
<tr><td nowrap><div align="left">raw theorem (B1 theorem)</div></td><td nowrap><div align="left">return, no baseline</div></td><td nowrap><div align="left">step size $\alpha$</div></td><td nowrap><div align="left">none</div></td></tr>
<tr><td nowrap><div align="left"><b>REINFORCE</b> (this session)</div></td><td nowrap><div align="left">return-to-go minus baseline</div></td><td nowrap><div align="left">step size $\alpha$</div></td><td nowrap><div align="left">moving average of returns</div></td></tr>
<tr><td colspan="4"><div align="left"><b>Preview: what the next sessions change</b></div></td></tr>
<tr><td nowrap><div align="left">actor-critic (B3 critic)</div></td><td nowrap><div align="left"><b>TD error from a critic</b></div></td><td nowrap><div align="left">step size $\alpha$</div></td><td nowrap><div align="left"><b>the critic</b></div></td></tr>
<tr><td nowrap><div align="left">PPO (B4 step size)</div></td><td nowrap><div align="left">advantage from a critic</div></td><td nowrap><div align="left"><b>clipped ratio</b></div></td><td nowrap><div align="left">the critic</div></td></tr>
<tr><td nowrap><div align="left">GRPO (Part C)</div></td><td nowrap><div align="left">group-relative advantage</div></td><td nowrap><div align="left">clipped ratio</div></td><td nowrap><div align="left"><b>mean of a group of episodes</b></div></td></tr>
</tbody></table>

- in the preview rows, bold marks the cell that is new
- the preview rows are confirmed session by session
- "step size $\alpha$" in slot 2 is the plain gradient step


## Aside: theoretical versus practical REINFORCE

<table>
<thead><tr><th nowrap><div align="left">Feature</div></th><th nowrap><div align="left">Theoretical REINFORCE</div></th><th nowrap><div align="left">Practical (batch, GRPO-like)</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">rollouts per prompt</div></td><td nowrap><div align="left">1</div></td><td nowrap><div align="left">several (for example $8$)</div></td></tr>
<tr><td nowrap><div align="left">baseline</div></td><td nowrap><div align="left">global moving average (historical)</div></td><td nowrap><div align="left">group mean (immediate)</div></td></tr>
<tr><td nowrap><div align="left">variance</div></td><td nowrap><div align="left">very high (small learning rates are usual)</div></td><td nowrap><div align="left">reduced by sampling more</div></td></tr>
<tr><td nowrap><div align="left">update frequency</div></td><td nowrap><div align="left">every trajectory</div></td><td nowrap><div align="left">after a group of trajectories</div></td></tr>
<tr><td nowrap><div align="left">compute per update</div></td><td nowrap><div align="left">low</div></td><td nowrap><div align="left">high (heavy generation)</div></td></tr>
<tr><td nowrap><div align="left">analogy</div></td><td nowrap><div align="left">your lifetime golf average</div></td><td nowrap><div align="left">ten swings from the same spot</div></td></tr>
</tbody></table>


# What we established, and what it sets up

- **Established**
    - the raw return is an unbiased but very noisy weight
        - it contains the shared part
        - it is one sample
        - it gives all steps the same credit
    - a baseline that does not depend on the action leaves the expected gradient unchanged and reduces the variance (the zero-mean fact)
    - the **advantage** is the return of one action relative to the state's average return
    - the unified form is $\E[\sum \nabla_\theta \log \pi_\theta \; \advseq]$
    - **REINFORCE** is the theorem estimated by sampling, with a moving-average baseline and no critic
        - it is still noisy
- **Sets up**
    - **the limitation:** REINFORCE's weight is still one sampled return, so each step is noisy
    - next session: replace part of the sampled return by a learned estimate, a **critic**: actor-critic
    - then slot 2, how far one update may move (PPO), then in Part C slot 3 without a critic (GRPO)
